In [1]:
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset, CSGridMLM_collate_fn
from torch.utils.data import DataLoader
from models import SEFiLMModel, ContrastiveSpaceModel, contrastive_normalized_loss
import torch
from torch.nn import CrossEntropyLoss
import os
import pickle
from data_utils import ContrastiveCollator
from train_utils import make_mixed_batch, full_to_partial_masking
from evaluation_utils import evaluate_contrastive_convergence
from tqdm import tqdm
from pprint import pprint

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
source_name = 'lstm'
shared_dim = 64
batch_size = 16
device_name = 'cuda:0'

In [3]:
source_key = source_name + '_embeddings'

train_path = 'data/contrastive_dataset/CA_train.pickle'
val_path = 'data/contrastive_dataset/CA_test.pickle'

with open(train_path, 'rb') as f:
    train_dataset = pickle.load(f)
with open(val_path, 'rb') as f:
    val_dataset = pickle.load(f)

source_dim = train_dataset[0][source_key].shape[0]
transformer_dim = train_dataset[0]['transformer_embeddings'].shape[0]

print(source_name, ' - source_dim: ', source_dim)
print('transformer', ' - transformer_dim: ', transformer_dim)
print('shared', ' - shared_dim: ', shared_dim)

collator = ContrastiveCollator(pad_id=0)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collator)

if device_name == 'cpu':
    device = torch.device('cpu')
else:
    if torch.cuda.is_available():
        device = torch.device(device_name)
    else:
        print('Selected device not available: ' + device_name)
# end device selection

contrastive_model = ContrastiveSpaceModel(source_dim, transformer_dim, shared_dim=shared_dim)
checkpoint = torch.load(f'saved_models/contrastive/{source_name}.pt', map_location=device_name)
contrastive_model.load_state_dict(checkpoint)
contrastive_model.to(device)

# contrastive_loss_fn = contrastive_normalized_loss
contrastive_loss_fn = torch.nn.MSELoss()

tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

mask_token_id = tokenizer.mask_token_id
bar_token_id = tokenizer.bar_token_id

logits_loss_fn =CrossEntropyLoss(ignore_index=-100)

transformer_film_model = SEFiLMModel(
    chord_vocab_size=len(tokenizer.vocab),
    d_model=512,
    nhead=8,
    num_layers=8,
    grid_length=80,
    pianoroll_dim=tokenizer.pianoroll_dim,
    guidance_dim=shared_dim,
    device=device,
)
checkpoint = torch.load(f'saved_models/film/{source_name}.pt', map_location=device_name)
transformer_film_model.load_state_dict(checkpoint)
transformer_film_model.to(device)

transformer_lacta_model = SEFiLMModel(
    chord_vocab_size=len(tokenizer.vocab),
    d_model=512,
    nhead=8,
    num_layers=8,
    grid_length=80,
    pianoroll_dim=tokenizer.pianoroll_dim,
    guidance_dim=shared_dim,
    device=device,
)
checkpoint = torch.load(f'saved_models/lacta/{source_name}.pt', map_location=device_name)
transformer_lacta_model.load_state_dict(checkpoint)
transformer_lacta_model.to(device)

transformer_lacta_full_model = SEFiLMModel(
    chord_vocab_size=len(tokenizer.vocab),
    d_model=512,
    nhead=8,
    num_layers=8,
    grid_length=80,
    pianoroll_dim=tokenizer.pianoroll_dim,
    guidance_dim=shared_dim,
    device=device,
)
checkpoint = torch.load(f'saved_models/lacta_full/{source_name}.pt', map_location=device_name)
transformer_lacta_full_model.load_state_dict(checkpoint)
transformer_lacta_full_model.to(device)

lstm  - source_dim:  128
transformer  - transformer_dim:  512
shared  - shared_dim:  64


SEFiLMModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (dropout): Dropout(p=0.3, inplace=False)
  (encoder_layers): ModuleList(
    (0-7): 8 x TransformerEncoderLayerWithFiLM(
      (layer): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
      (film): FiLMAdapter(
        (gamma): Linear(in_features=64, out_features=512, bias=True)
        (beta): Line

In [14]:
film_eval_dict = evaluate_contrastive_convergence(
        transformer_film_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device
)

pprint(film_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 55.31batch/s, acc=0.68, floss=0.0138, hloss=0.0132, loss=0.0236] 

{'val_foreign_acc': 0.5829427083333333,
 'val_foreign_logits_loss': 1.9424300541480382,
 'val_foreign_loss': 0.013762278385305157,
 'val_home_acc': 0.7407714843749998,
 'val_home_logits_loss': 0.7891940797368685,
 'val_home_loss': 0.013205878552980721,
 'val_no2foreign_loss': 0.031028679727266233,
 'val_no2home_acc': 0.6194173177083333,
 'val_no2home_logits_loss': 1.4161849009493988,
 'val_no2home_loss': 0.02925966839150836}


In [15]:
lacta_eval_dict = evaluate_contrastive_convergence(
        transformer_lacta_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device
)

pprint(lacta_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 53.74batch/s, acc=0.68, floss=0.0027, hloss=0.00247, loss=0.0168]  

{'val_foreign_acc': 0.585205078125,
 'val_foreign_logits_loss': 1.941029558579127,
 'val_foreign_loss': 0.0027039266375747197,
 'val_home_acc': 0.7415852864583335,
 'val_home_logits_loss': 0.7867728061974049,
 'val_home_loss': 0.0024690498054648438,
 'val_no2foreign_loss': 0.030993159821567435,
 'val_no2home_acc': 0.6183593750000002,
 'val_no2home_logits_loss': 1.4153600583473842,
 'val_no2home_loss': 0.029253881191834807}


In [6]:
lacta_full_eval_dict = evaluate_contrastive_convergence(
        transformer_lacta_full_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device
)

pprint(lacta_full_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 55.44batch/s, acc=0.603, floss=0.032, hloss=0.0317, loss=0.032]  

{'val_foreign_acc': 0.5914713541666667,
 'val_foreign_logits_loss': 1.7842614365120728,
 'val_foreign_loss': 0.03201592833890269,
 'val_home_acc': 0.5874837239583331,
 'val_home_logits_loss': 1.8245837092399597,
 'val_home_loss': 0.03165225773894539,
 'val_no2foreign_loss': 0.0310680199181661,
 'val_no2home_acc': 0.6179687500000001,
 'val_no2home_logits_loss': 1.4239866808056831,
 'val_no2home_loss': 0.02920646018659075}


In [16]:
film_eval_dict = evaluate_contrastive_convergence(
        transformer_film_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device,
        interpolate=True
)

pprint(film_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 53.58batch/s, acc=0.68, floss=0.0124, hloss=0.0132, loss=0.0208] 

{'val_foreign_acc': 0.6266438802083334,
 'val_foreign_logits_loss': 1.5551759141186874,
 'val_foreign_loss': 0.012439932111495485,
 'val_home_acc': 0.7422851562499999,
 'val_home_logits_loss': 0.7894535182664791,
 'val_home_loss': 0.013154342925796906,
 'val_no2foreign_loss': 0.023066819024582703,
 'val_no2home_acc': 0.6175292968749999,
 'val_no2home_logits_loss': 1.4166913678248723,
 'val_no2home_loss': 0.029146964584166806}


In [17]:
lacta_eval_dict = evaluate_contrastive_convergence(
        transformer_lacta_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device,
        interpolate=True
)

pprint(lacta_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 54.46batch/s, acc=0.681, floss=0.0047, hloss=0.00247, loss=0.0137] 

{'val_foreign_acc': 0.6228841145833334,
 'val_foreign_logits_loss': 1.5675348527729511,
 'val_foreign_loss': 0.004702956590335816,
 'val_home_acc': 0.7416666666666668,
 'val_home_logits_loss': 0.7886902118722597,
 'val_home_loss': 0.0024667169394282005,
 'val_no2foreign_loss': 0.023039839618528884,
 'val_no2home_acc': 0.6198567708333332,
 'val_no2home_logits_loss': 1.417366836220026,
 'val_no2home_loss': 0.029154314659535885}


In [9]:
lacta_full_eval_dict = evaluate_contrastive_convergence(
        transformer_lacta_full_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device,
        interpolate=True
)

pprint(lacta_full_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 55.17batch/s, acc=0.603, floss=0.0239, hloss=0.0316, loss=0.0282]

{'val_foreign_acc': 0.6135579427083333,
 'val_foreign_logits_loss': 1.6150584854185581,
 'val_foreign_loss': 0.023859085515141487,
 'val_home_acc': 0.5886555989583333,
 'val_home_logits_loss': 1.8252417296171188,
 'val_home_loss': 0.031629024694363274,
 'val_no2foreign_loss': 0.023042217440282304,
 'val_no2home_acc': 0.617578125,
 'val_no2home_logits_loss': 1.4266609996557236,
 'val_no2home_loss': 0.029275004713175196}


In [18]:
film_eval_dict = evaluate_contrastive_convergence(
        transformer_film_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device,
        extrapolate=True
)

pprint(film_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 53.66batch/s, acc=0.68, floss=1.51, hloss=0.0132, loss=0.793] 

{'val_foreign_acc': 0.024479166666666673,
 'val_foreign_logits_loss': 6.318072646856308,
 'val_foreign_loss': 1.5117654204368591,
 'val_home_acc': 0.7408854166666666,
 'val_home_logits_loss': 0.78793632487456,
 'val_home_loss': 0.013202330543814847,
 'val_no2foreign_loss': 1.5767173195878665,
 'val_no2home_acc': 0.6183268229166666,
 'val_no2home_logits_loss': 1.4225542669494946,
 'val_no2home_loss': 0.02928064107739677}


In [19]:
lacta_eval_dict = evaluate_contrastive_convergence(
        transformer_lacta_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device,
        extrapolate=True
)

pprint(lacta_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 55.01batch/s, acc=0.679, floss=1.34, hloss=0.00246, loss=0.733]

{'val_foreign_acc': 0.022851562500000002,
 'val_foreign_logits_loss': 6.223151485125224,
 'val_foreign_loss': 1.3371018767356873,
 'val_home_acc': 0.7400065104166668,
 'val_home_logits_loss': 0.7879591081291437,
 'val_home_loss': 0.002463159068914441,
 'val_no2foreign_loss': 1.5756885980566342,
 'val_no2home_acc': 0.6176269531250002,
 'val_no2home_logits_loss': 1.4224063456058502,
 'val_no2home_loss': 0.029257085911619168}


In [12]:
lacta_full_eval_dict = evaluate_contrastive_convergence(
        transformer_lacta_full_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device,
        extrapolate=True
)

pprint(lacta_full_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 55.56batch/s, acc=0.603, floss=1.58, hloss=0.0316, loss=0.795]

{'val_foreign_acc': 0.029606119791666653,
 'val_foreign_logits_loss': 5.709452132383983,
 'val_foreign_loss': 1.5842213183641434,
 'val_home_acc': 0.5883463541666666,
 'val_home_logits_loss': 1.8232224099338055,
 'val_home_loss': 0.03164093068335205,
 'val_no2foreign_loss': 1.5740982343753178,
 'val_no2home_acc': 0.617919921875,
 'val_no2home_logits_loss': 1.4190046948691208,
 'val_no2home_loss': 0.02923531283158809}


### Linearity and curvature of the contrastive space

Interpolation can occur either before or after the contrastive space. To do that, we examine how much the interpolated values of random pairs before the contrastive space fall close to the respective interpolated values after the contrastive space.

In [13]:
diff = 0

for batch in val_loader:
    # mix the batch twice
    mixed_batch_1 = make_mixed_batch(batch, source_key)
    mixed_batch_2 = make_mixed_batch(mixed_batch_1, source_key)
    # get interpolated values
    before_contrastive = (mixed_batch_1[source_key].to(device) + mixed_batch_2[source_key].to(device))/2
    # get contrastive image
    pre_contrastive_image = contrastive_model.source_proj(before_contrastive.to(device))
    # get each image and mix after
    contra_image_1 = contrastive_model.source_proj(mixed_batch_1[source_key].to(device))
    contra_image_2 = contrastive_model.source_proj(mixed_batch_2[source_key].to(device))
    # mix after
    post_contrastive_mix = (contra_image_1 + contra_image_2)/2
    
    diff += torch.norm((pre_contrastive_image - post_contrastive_mix))/(torch.norm(contra_image_2)*len(batch))
diff /= len(val_loader)
print(diff)

tensor(0.0853, device='cuda:0', grad_fn=<DivBackward0>)
